## Atelier 0 — Mise en route (5 min)
Les **paramètres** sont regroupés en tête de notebook : c'est une règle d'auditabilité (chemins et seuils ne se cachent pas au milieu du code).

In [ ]:
# --- PARAMÈTRES (toujours en tête du notebook) ---------------------------
CHEMIN_DONNEES = "portefeuille_vie.csv"
DATE_REF = "2025-01-01"     # début de la période d'observation (rachat_12m = rachat sur l'exercice 2025)
SEUIL_EFFECTIF = 100        # en dessous, un taux par segment est jugé peu fiable

# --- IMPORTS --------------------------------------------------------------
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)


# --- OUTIL D'AUTO-CONTRÔLE (ne pas modifier) ------------------------------
def verifier(libelle, test):
    """Affiche ✅ ou ❌ selon le résultat de `test` (une fonction sans argument)."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + libelle + ("" if ok else "   → à compléter ou à revoir"))

---
# Atelier C — Agréger : du tableau croisé dynamique au `groupby` (≈ 22 min)
*Rappel théorique : « les taux actuariels sont des moyennes par groupe », somme (stock) vs moyenne (taux) (slides « groupby actuariel » et « matrice de travail »).*

Vous construisez déjà ces tableaux dans Excel. Voici la traduction :

| Dans Excel | En pandas | Dans cet atelier |
| --- | --- | --- |
| Tableau croisé dynamique | `df.groupby("col").agg(...)` | C2, C3 |
| `NB.SI.ENS` / `SOMME.SI.ENS` | `groupby` + `count` / `sum` | C2, C3 |
| Contrôle « somme des segments = total » | `verifier(...)` / `assert` | C4 |
| `RECHERCHEV` / `XLOOKUP` | `df.merge(table, on="clé", how="left")` | C6 |
| Colonne calculée | `df["nouvelle"] = ...` | C5 |
| Intervalle de confiance (feuille à part) | formule vectorisée | C7 |

L'avantage : le calcul est **écrit, rejouable et relisible** par un validateur, là où un TCD ne laisse aucune trace de ses paramètres.

### C1 ·  Le taux de rachat global
Le taux de rachat est une **moyenne conditionnelle** : ici, la moyenne d'une variable 0/1 est une proportion.
<details><summary>💡 Indice</summary><code>df.colonne.mean()</code>.</details>

In [ ]:
taux_global = df.....mean()
print(f"Taux de rachat global : {taux_global:.2%} sur {len(df):,} contrats")

verifier("taux global ≈ 4,3 %", lambda: abs(taux_global - 0.043) < 0.0005)

### C2 · Taux de rachat par tranche d'âge
Découpez l'âge en tranches (`pd.cut`), puis calculez effectif et taux par tranche. Le profil attendu est **en U** (jeunes : besoin de liquidité ; seniors : arbitrage, succession).
<details><summary>💡 Indice</summary>Dans <code>.agg(nom=("colonne", "fonction"))</code>, la fonction est un texte : <code>"count"</code>, <code>"mean"</code>, <code>"sum"</code>, <code>"median"</code>.</details>

In [ ]:
bornes = [18, 30, 45, 60, 70, 101]
etiquettes = ["18-29", "30-44", "45-59", "60-69", "70+"]
df["tranche_age"] = pd.cut(df.age, bins=bornes, labels=etiquettes, right=False)

taux_age = (df.groupby("tranche_age", observed=True)
              .agg(effectif=("id_contrat", "count"),
                   taux_rachat=("rachat_12m", ...)))
taux_age

In [ ]:
verifier("5 tranches", lambda: len(taux_age) == 5)
verifier("la tranche 70+ est la plus élevée (≈ 8,3 %)", lambda: taux_age.taux_rachat.idxmax() == "70+" and abs(taux_age.loc["70+", "taux_rachat"] - 0.0828) < 0.001)
verifier("profil en U : 30-44 plus bas que 18-29 et 70+", lambda: taux_age.loc["30-44", "taux_rachat"] < min(taux_age.loc["18-29", "taux_rachat"], taux_age.loc["70+", "taux_rachat"]))

### C3 ·  Stock et taux par génération
Pour chaque génération : effectif, taux de rachat, **encours total** (un stock : on le **somme**) et **encours moyen**. Ne confondez pas somme (stock) et moyenne (taux).

In [ ]:
taux_gen = (df.groupby(...)
              .agg(effectif=("id_contrat", "count"),
                   taux_rachat=("rachat_12m", "mean"),
                   encours_total=("encours", ...),
                   encours_moyen=("encours", ...)))
taux_gen.round(4)

In [ ]:
verifier("10 générations", lambda: len(taux_gen) == 10)
verifier("effectifs : somme = 5 000", lambda: taux_gen.effectif.sum() == 5000)
verifier("encours total 2024 ≈ 39,6 M€", lambda: abs(taux_gen.loc[2024, "encours_total"] / 1e6 - 39.6) < 0.1)

### C4 · Le contrôle de rapprochement : somme des segments = total ?
Réflexe hérité de la production actuarielle. Vérifiez que le tableau par tranche d'âge couvre bien **tous** les contrats. Si ce n'est pas le cas, trouvez pourquoi et corrigez.
<details><summary>💡 Indice</summary>Les contrats dont l'âge est <code>NaN</code> n'appartiennent à aucune tranche : <code>groupby</code> les ignore silencieusement. Solution : leur créer une modalité explicite avec <code>.cat.add_categories("Non renseigné")</code> puis <code>.fillna("Non renseigné")</code>.</details>

In [ ]:
# 1. Combien de contrats manquent à l'appel ?
ecart_effectif = len(df) - taux_age.effectif....
print("Contrats absents du tableau par tranche d'âge :", ecart_effectif)

# 2. Correction : une modalité explicite plutôt qu'un oubli silencieux
df["tranche_age"] = (df["tranche_age"]
                     .cat.add_categories(...)
                     .fillna(...))

taux_age = (df.groupby("tranche_age", observed=True)
              .agg(effectif=("id_contrat", "count"),
                   taux_rachat=("rachat_12m", "mean")))
taux_age

In [ ]:
verifier("l'écart détecté était de 81 contrats", lambda: ecart_effectif == 81)
verifier("rapprochement OK : somme des effectifs = 5 000", lambda: taux_age.effectif.sum() == len(df))
verifier("modalité « Non renseigné » présente", lambda: "Non renseigné" in taux_age.index)

### C5 ·  Taux en nombre vs taux en montant
En assurance vie, un rachat de 200 000 € ne pèse pas comme 10 rachats de 2 000 €. Calculez, par tranche d'âge, le **taux en montant** = encours racheté ÷ encours total, et comparez-le au taux en nombre.
<details><summary>💡 Indice</summary>Créez d'abord une colonne <code>encours_rachete = encours × rachat_12m</code> (l'encours des seuls contrats rachetés), sommez-la par tranche, puis divisez par la somme des encours.</details>

In [ ]:
df["encours_rachete"] = df.encours * df.rachat_12m

par_tranche = (df.groupby("tranche_age", observed=True)
                 .agg(encours_total=("encours", "sum"),
                      encours_rachete=("encours_rachete", "sum")))
par_tranche["taux_montant"] = par_tranche.encours_rachete / ...

comparaison = taux_age[["taux_rachat"]].join(par_tranche[["taux_montant"]])
taux_global_montant = df.encours_rachete.sum() / df.encours.sum()
print(f"Taux global en nombre : {taux_global:.2%} | en montant : {taux_global_montant:.2%}")
comparaison.round(4)

In [ ]:
verifier("taux global en montant ≈ 3,9 %", lambda: abs(taux_global_montant - 0.0391) < 0.0005)
verifier("tranche 45-59 : le taux en montant est inférieur au taux en nombre", lambda: comparaison.loc["45-59", "taux_montant"] < comparaison.loc["45-59", "taux_rachat"])

**À discuter à deux (2 min)** : sur cet échantillon, le taux en montant est plus faible que le taux en nombre. Que cela signifie-t-il sur la taille des contrats rachetés ? Pour un calcul de Best Estimate ou une analyse ALM, lequel des deux taux faut-il projeter, et pourquoi ?

### C6 ·  Le `RECHERCHEV` de pandas : rattacher une table de référence
Rattachez à chaque contrat le **taux de revalorisation servi** de sa génération (table de référence **fictive** ci-dessous). Le piège classique : une jointure mal faite **duplique ou perd des lignes**. D'où le contrôle du nombre de lignes avant / après.
<details><summary>💡 Indice</summary><code>df.merge(table, on="cle", how="left")</code> : « left » conserve toutes les lignes de <code>df</code>, comme un RECHERCHEV qui ne supprime jamais de ligne.</details>

In [ ]:
taux_servi = pd.DataFrame({
    "generation": list(range(2015, 2025)),
    "taux_servi": [0.023, 0.021, 0.019, 0.018, 0.016, 0.014, 0.015, 0.017, 0.020, 0.024],   # valeurs FICTIVES
})

df = df.drop(columns="taux_servi", errors="ignore")    # permet de rejouer la cellule sans doublon de colonne
n_avant = len(df)
df = df.merge(taux_servi, on=..., how=...)

verifier("aucune ligne perdue ni dupliquée", lambda: len(df) == n_avant)
verifier("aucun taux servi manquant", lambda: df.taux_servi.notna().all())
df[["id_contrat", "generation", "taux_servi"]].head()

### C7 ·  Peut-on se fier à un taux par segment ? Mesurer l'incertitude
Un taux de rachat est une **estimation**. Pour une proportion `p` observée sur `n` contrats, la demi-largeur approchée de l'intervalle de confiance à 95 % est `1,96 × √(p(1−p)/n)`.
1. Écrivez la fonction `demi_largeur_ic(p, n)`.
2. Comparez : **4,3 % sur 5 000 contrats** vs **4,3 % sur 40 contrats**.
3. Ajoutez l'IC et un indicateur de fiabilité (`effectif ≥ SEUIL_EFFECTIF`) au tableau par tranche d'âge, puis comptez les cellules **tranche × produit** sous le seuil.
<details><summary>💡 Indice</summary>Racine carrée vectorisée : <code>np.sqrt(...)</code>. Pour compter des « vrai » dans une série booléenne : <code>(serie &lt; seuil).sum()</code>. <i>(L'approximation normale est grossière pour les très petits effectifs : à retenir pour la suite.)</i></details>

In [ ]:
def demi_largeur_ic(p, n):
    """Demi-largeur de l'IC à 95 % d'une proportion p observée sur n contrats (approximation normale)."""
    return ...

print(f"4,3 % sur 5 000 contrats : ± {demi_largeur_ic(0.043, 5000):.2%}")
print(f"4,3 % sur    40 contrats : ± {demi_largeur_ic(0.043, 40):.2%}")

taux_age["ic95"] = demi_largeur_ic(taux_age.taux_rachat, taux_age.effectif)
taux_age["fiable"] = taux_age.effectif ... SEUIL_EFFECTIF

croise = (df[df.tranche_age != "Non renseigné"]
            .groupby(["tranche_age", "produit"], observed=True)
            .agg(effectif=("id_contrat", "count"), taux_rachat=("rachat_12m", "mean")))
nb_cellules_fragiles = (croise.effectif ... SEUIL_EFFECTIF).sum()
print(f"\nCellules tranche × produit sous {SEUIL_EFFECTIF} contrats : {nb_cellules_fragiles} sur {len(croise)}")
taux_age.round(4)

In [ ]:
verifier("IC à n = 5 000 ≈ ± 0,56 pt", lambda: abs(demi_largeur_ic(0.043, 5000) - 0.0056) < 0.0005)
verifier("IC à n = 40 ≈ ± 6,3 pts : plus large que le taux lui-même", lambda: abs(demi_largeur_ic(0.043, 40) - 0.0629) < 0.001)
verifier("5 cellules fragiles sur 15", lambda: nb_cellules_fragiles == 5)